# Drawdown Analysis 

This notebook provides functions to analyze drawdowns in financial time series.

It includes:
- Drawdown calculation
- Statistical analysis
- Event detection
- Visualization
- MAR ratio evaluation

In [ ]:
"""
Technical setup and data preparation.
"""

from src.technical_setup import *  # Assumes this provides `stock`, `TICKER`, plt, etc.

# Ensure we have a proper Close column
stock["Close"] = stock[TICKER]

## Drawdown Calculation

This function computes the drawdown of a price series relative to its historical peak.

**Key idea:**
- Track the cumulative maximum
- Measure percentage decline from that peak


In [ ]:
def calculate_drawdown(stock: pd.DataFrame) -> pd.DataFrame:
    """

    Parameters
    ----------
    stock : pd.DataFrame
        DataFrame containing at least a 'Close' or 'ACM' column.

    Returns
    -------
    pd.DataFrame
        DataFrame with additional columns:
        - rolling_max: cumulative maximum of price
        - drawdown: percentage drawdown from peak
    """
    df = stock.copy()

    df["rolling_max"] = df["Close"].cummax()
    df["drawdown"] = (df["Close"] - df["rolling_max"]) / df["rolling_max"]

    return df

## Drawdown Statistics

Computes summary statistics of the drawdown series.

**Metrics:**
- Maximum drawdown (worst loss)
- Mean drawdown
- Median drawdown

Useful for quick risk assessment.

In [ ]:
def drawdown_analysis(drawdown_stock: pd.DataFrame) -> tuple:
    """

    Parameters
    ----------
    drawdown_stock : pd.DataFrame
        DataFrame containing a 'drawdown' column.

    Returns
    -------
    tuple
        (max_drawdown, mean_drawdown, median_drawdown)
    """
    max_drawdown = drawdown_stock["drawdown"].min()
    mean_drawdown = drawdown_stock["drawdown"].mean()
    median_drawdown = drawdown_stock["drawdown"].median()

    return max_drawdown, mean_drawdown, median_drawdown

## Drawdown Visualization

Plots:
- Price series
- Corresponding drawdown

Helps visually identify major drawdown periods.

In [ ]:
def plot_drawdown(drawdown_stock: pd.DataFrame, 
                  save_path: str = None
                  ) -> None:
    """
    Plot price and drawdown over time.

    Parameters
    ----------
    drawdown_stock : pd.DataFrame
        DataFrame containing 'Close' and 'drawdown'.
    save_path : str, optional
        Path to save the plot, by default None
    """
    fig, ax = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

    drawdown_stock["Close"].plot(ax=ax[0], title="Price")
    drawdown_stock["drawdown"].plot(ax=ax[1], title="Drawdown", color="red")

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


## Drawdown Event Detection

Identifies individual drawdown periods (peak → trough → recovery).

Each event includes:
- Start date
- Trough date
- End date
- Maximum drawdown
- Duration

This is useful for deeper risk analysis beyond summary statistics.

In [ ]:
def drawdown_events(stock: pd.DataFrame) -> pd.DataFrame:
    """

    A drawdown event starts when the price falls below its previous peak
    and ends when the drawdown period stops (last negative drawdown).

    Parameters
    ----------
    stock : pd.DataFrame
        Input price DataFrame.

    Returns
    -------
    pd.DataFrame
        Event-level statistics:
        - start : first date of drawdown
        - trough : date of maximum drawdown (worst loss)
        - end : last date before recovery
        - max_dd : maximum drawdown value
        - duration : length of drawdown (in periods)
    """
    df = calculate_drawdown(stock).sort_index()

    in_drawdown = df["drawdown"] < 0
    event_id = (in_drawdown != in_drawdown.shift()).cumsum()

    dd_df = df[in_drawdown].copy()
    dd_df["event"] = event_id[in_drawdown]

    events = dd_df.groupby("event").agg(
        start=("drawdown", lambda x: x.index[0]),
        trough=("drawdown", "idxmin"),
        end=("drawdown", lambda x: x.index[-1]),
        max_dd=("drawdown", "min"),
        duration=("drawdown", "count"),
    )

    return events

## Duration vs Severity Analysis

Scatter plot showing:
- Duration of drawdowns
- Maximum drawdown per event

Helps identify whether longer drawdowns tend to be more severe.

In [ ]:
def plot_drawdown_scatter(events: pd.DataFrame, 
                          save_path: str = None
                          ) -> None:
    """
    Scatter plot of drawdown duration vs. maximum drawdown.

    Parameters
    ----------
    events : pd.DataFrame
        Output from drawdown_events().
    save_path : str, optional
        Path to save the plot, by default None
    """
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.scatter(events["duration"], events["max_dd"], alpha=0.7)

    ax.set_title("Drawdown: Duration vs Max Drawdown")
    ax.set_xlabel("Duration (Days)")
    ax.set_ylabel("Max Drawdown")

    ax.axhline(0, linestyle="--")
    ax.grid(True)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.show()

## Drawdown Duration Analysis

Provides two views:
- Time series of drawdown durations
- Histogram of duration distribution

Useful for understanding how long drawdowns typically last.

In [ ]:
def plot_duration_analysis(events: pd.DataFrame,
                           save_path: str = None
                           ) -> None:
    """
    Analyze drawdown durations over time and distribution.

    Parameters
    ----------
    events : pd.DataFrame
        Output from drawdown_events().
    save_path : str, optional
        Path to save the plot, by default None
    """
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    # Time series of durations
    ax[0].plot(events["duration"], marker="o")
    ax[0].set_title("Drawdown Duration Over Time")
    ax[0].set_xlabel("Event")
    ax[0].set_ylabel("Days")
    ax[0].grid(True)

    # Histogram
    ax[1].hist(events["duration"], bins=20)
    ax[1].set_title("Duration Distribution")
    ax[1].set_xlabel("Days")
    ax[1].set_ylabel("Frequency")
    ax[1].grid(True)

    plt.tight_layout()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## MAR Ratio Analysis

Calculates the MAR ratio:

    MAR = (CAGR - Risk-Free Rate) / Max Drawdown

This metric evaluates risk-adjusted performance:
- Higher = better return per unit of drawdown risk

In [ ]:
def mar_ratio_analysis(
    stock: pd.DataFrame,
    events: pd.DataFrame,
    risk_free_rate: float = 0.04
) -> dict:
    """

    Parameters
    ----------
    stock : pd.DataFrame
        Price series with 'Close'.
    events : pd.DataFrame
        Drawdown events.
    risk_free_rate : float, optional
        Annual risk-free rate, by default 0.04.

    Returns
    -------
    dict
        Dictionary with:
        - CAGR
        - Max Drawdown
        - MAR Ratio
    """
    start_value = stock["Close"].iloc[0]
    end_value = stock["Close"].iloc[-1]

    years = len(stock) / 252  # Trading days assumption
    cagr = (end_value / start_value) ** (1 / years) - 1

    max_drawdown = events["max_dd"].min()

    if max_drawdown == 0:
        return None

    mar_ratio = (cagr - risk_free_rate) / abs(max_drawdown)

    return {
        "CAGR": cagr,
        "Max Drawdown": max_drawdown,
        "MAR Ratio": mar_ratio,
    }


In [ ]:
# Run full analysis pipeline

drawdown_stock = calculate_drawdown(stock)
drawdown_stats = drawdown_analysis(drawdown_stock)
events = drawdown_events(stock)
mar_results = mar_ratio_analysis(stock, events)

print(f"Max Drawdown: {drawdown_stats[0]:.2%}")
print(f"Mean Drawdown: {drawdown_stats[1]:.2%}")
print(f"Median Drawdown: {drawdown_stats[2]:.2%}")

print(f"CAGR: {mar_results['CAGR']:.2%}")
print(f"MAR Ratio: {mar_results['MAR Ratio']:.2f}")

plot_drawdown(drawdown_stock,
              save_path=f"figures/{TICKER}_drawdown_and_price.png")
plot_drawdown_scatter(events,
                      save_path=f"figures/{TICKER}_drawdown_scatter.png")
plot_duration_analysis(events,
                       save_path=f"figures/{TICKER}_duration_analysis.png")